# Generate the FD001 detector ROC figure from archived predictions.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT


In [ ]:
import json
PLAN = load_json('assets/11/02_plan.json')
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import bisect, csv, gzip, hashlib, itertools, json, math, statistics, traceback

METHODS = ['DBSCAN','IF','LOF','OC-SVM','kNN']
PAIRS = [(s,f) for s in (42,43,44) for f in range(1,11)]
GRID = [i / 1000 for i in range(1001)]

def require(condition, message):
    if not condition:
        raise ValueError(message)

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1048576), b''):
            h.update(chunk)
    return h.hexdigest()

def load(path):
    return json.loads(path.read_text(encoding='utf-8'))

def canonical(data):
    encoded = json.dumps(data,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False)
    return hashlib.sha256(encoded.encode()).hexdigest()

def roc_and_ap(y, scores):

    ranked = sorted(zip(scores,y), key=lambda item:item[0], reverse=True)
    p = sum(y)
    n = len(y) - p
    require(p > 0 and n > 0,'Her fold iki sınıf içermeli')
    tp = fp = 0
    xs, ys = [0.0], [0.0]
    ap = auc = 0.0
    for _, group in itertools.groupby(ranked, key=lambda item:item[0]):
        labels = [v for _,v in group]
        hits = sum(labels)
        tp += hits
        fp += len(labels) - hits
        xx, yy = fp/n, tp/p
        ap += (hits/p) * tp/(tp+fp)
        auc += (xx-xs[-1]) * (yy+ys[-1])/2
        xs.append(xx); ys.append(yy)

    curve = []
    for fpr in GRID:
        i = bisect.bisect_right(xs,fpr)-1
        if xs[i] == fpr:
            curve.append(ys[i])
        else:
            w = (fpr-xs[i])/(xs[i+1]-xs[i])
            curve.append(ys[i]+w*(ys[i+1]-ys[i]))
    return auc,ap,curve

def read_predictions(folder,record,name,methods):
    p = folder/name
    require(digest(p)==record['output_sha256'][name],f'Tahmin dosyası değişmiş: {p}')
    opener = gzip.open if name.endswith('.gz') else open
    by_method = {m:{} for m in methods}
    with opener(p,'rt',encoding='utf-8',newline='') as stream:
        for row in csv.DictReader(stream):
            m = row['method']
            if row['representation']!='AE_latent21' or m not in by_method:
                continue
            key = row['sample_key']
            require(key and key not in by_method[m],f'Tekrarlanan örnek: {m} {key}')
            score = float(row['score'])
            y = int(row['late_life_le_30'])
            rul = int(row['RUL'])
            pred = int(row['prediction'])
            require(math.isfinite(score) and pred in (0,1) and y==int(rul<=30),'Geçersiz skor/etiket')
            by_method[m][key] = (int(row['UnitNumber']),int(row['Time']),rul,y,score,pred)
    return by_method

def export():
    root = PROJECT_DIR/'revision_outputs'

    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    folder = root/'11_FD001_Figure6_ROC'/stamp
    folder.mkdir(parents=True, exist_ok=False)
    output = folder/'figure6_roc_summary.json'
    report = dict(stage='11_FD001_Figure6_ROC',status='RUNNING',completed_pairs=0,methods=METHODS)
    try:
        cv = load(root/PLAN['cv_path'])
        require(canonical(cv)==PLAN['cv_hash'],'Onaylanmış 05B özetiyle uyuşmuyor')
        expected = {(r['seed'],r['fold'],r['method']):r for r in cv['fold_primary_results'] if r['representation']=='AE_latent21' and r['RUL_window']==30}
        require(set(expected)=={(s,f,m) for s,f in PAIRS for m in METHODS},'Referans kapsamı eksik')
        jobs = {(j['seed'],j['fold']):j for j in PLAN['jobs']}
        all_curves=defaultdict(list); checks=[]; signatures={}; engines_by_fold={}
        for seed,fold in PAIRS:
            job = jobs[seed,fold]
            orig = root/job['old_dir']
            old = load(orig/'fold_summary.json')
            require(old['status']=='PASS' and (old['seed'],old['fold'])==(seed,fold),'05 kimliği farklı')
            marker = root/job['new_marker']
            require(digest(marker)==job['new_marker_sha256'],'Tamamlama özeti değişmiş')
            new = load(marker)
            require(new['status'] in ('PASS_FOLD','PASS_PILOT') and (new['seed'],new['fold'])==(seed,fold),'05B/03B kimliği farklı')
            records = read_predictions(orig,old,'outer_predictions.csv',['IF','LOF','kNN'])
            records.update(read_predictions(marker.parent,new,'new_outer_predictions.csv.gz',['DBSCAN','OC-SVM']))
            reference = records['IF']; keys = sorted(reference)
            meta = [(key,)+reference[key][:4] for key in keys]
            signature = canonical(meta)
            if fold in signatures:require(signature==signatures[fold],'Seedler arasında dış motor/örnek değişmiş')
            signatures[fold]=signature
            engines_by_fold[fold]={reference[key][0] for key in keys}
            require(len(engines_by_fold[fold])==10,'FD001 dış fold motor sayısı farklı')
            for m in METHODS:
                values=records[m]; require(set(values)==set(keys),'Metotlar aynı dış örnekleri içermiyor')
                require(all(values[key][:4]==reference[key][:4] for key in keys),'Metotlar arasında örnek metadata farklı')
                y=[values[key][3] for key in keys]; scores=[values[key][4] for key in keys]; pred=[values[key][5] for key in keys]
                target=expected[seed,fold,m]
                for truth,hat,label in [(1,1,'TP'),(0,1,'FP'),(0,0,'TN'),(1,0,'FN')]:
                    require(sum(a==truth and b==hat for a,b in zip(y,pred))==target[label],f'{m}: {label} uyuşmuyor')
                auc,ap,curve=roc_and_ap(y,scores)
                require(abs(auc-target['ROC_AUC'])<=1e-10 and abs(ap-target['AP'])<=1e-10,f'{seed}/{fold}/{m}: AUC veya AP uyuşmuyor')
                all_curves[m].append(curve)
                checks.append(dict(seed=seed,fold=fold,method=m,rows=len(y),positive=sum(y),ROC_AUC=auc,AP=ap))
            report['completed_pairs']+=1
            if fold in (1,10):print(f'Seed {seed}, fold {fold} doğrulandı.',flush=True)
        require(len(set.union(*engines_by_fold.values()))==100,'Dış fold motorları ayrık değil')
        curves={}
        for m in METHODS:
            aucs=[r['ROC_AUC'] for r in checks if r['method']==m]
            mean_tpr=[statistics.mean(v) for v in zip(*all_curves[m])]
            grid_area=sum((b-a)*(u+v)/2 for a,b,u,v in zip(GRID,GRID[1:],mean_tpr,mean_tpr[1:]))
            mean_auc=statistics.mean(aucs)
            require(abs(grid_area-mean_auc)<=.001000001,'Ortalama eğri alanı interpolasyon toleransını aştı')
            curves[m]=dict(mean_tpr=mean_tpr,mean_exact_AUC=mean_auc,sd_exact_AUC=statistics.stdev(aucs),grid_curve_area=grid_area)
        report.update(status='PASS_FIGURE6_ROC',representation='AE_latent21',RUL_window=30,fold_checks=checks,fpr_grid=GRID,curves=curves,source05B_sha256=PLAN['cv_hash'],aggregation='30 equal-weight outer ROC curves on a common FPR grid. AUC labels use exact empirical fold AUCs. For drawing, prepend (0,0) to include any initial vertical segment.')
    except Exception as exc:
        report.update(status='REVIEW_REQUIRED',error=str(exc),traceback=traceback.format_exc())
        print(report['traceback'])
    output.write_text(json.dumps(report,ensure_ascii=False,allow_nan=False,indent=2),encoding='utf-8')
    print('DURUM:',report['status']);print('GÖNDERİLECEK DOSYA:',output)
    return report


In [ ]:
result = export()
